## Smoke Test — OpenAI LLM Connection (User 1696, 1 request)

Purpose: verify your OpenAI API key + model work end-to-end with the categorization prompt.

Cost controls (aligned to project limits):
- Only **one** LLM request (batch of up to 20 merchants)
- Uses **merchant-key caching** so repeated runs are cheaper
- Filters to **client_id = 1696**

Prereqs:
- `.env` contains `LLM_API_KEY` (or `OPENAI_API_KEY`) and `LLM_MODEL` (default: `gpt-4o-mini`)
- Run `data_processing/clean.ipynb` first to create `artifacts/transactions_enriched.json`


## Imports

In [2]:
from __future__ import annotations

import os
import sys
from pathlib import Path

import pandas as pd
from diskcache import Cache
from dotenv import load_dotenv


def find_project_root(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "data").is_dir() and (candidate / "artifacts").is_dir():
            return candidate
    raise FileNotFoundError("Could not locate project root containing data/ and artifacts/")


ROOT = find_project_root()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from data_processing.categorize_core import (
    DEFAULT_TAXONOMY,
    categorize_with_cache_and_fallback,
    make_openai_llm_call,
    merchant_fingerprint,
)
from model.analytics_core import project_root


## Config (keep this small for a cheap smoke test)

In [ ]:
load_dotenv(ROOT / ".env")

CLIENT_ID = 1696
CHUNKSIZE = 200_000
MAX_UNIQUE_MERCHANTS = 20  # 1 LLM request
CONF_THRESHOLD = 0.6

LLM_MODEL = os.environ.get("LLM_MODEL", "gpt-5.2")
BASE_URL = os.environ.get("OPENAI_BASE_URL") or os.environ.get("LLM_API_BASE")

root = project_root()
artifacts = root / "artifacts"
ndjson_path = artifacts / "transactions_enriched.json"
cache_dir = artifacts / "llm_cache"

print("client_id:", CLIENT_ID)
print("model:", LLM_MODEL)
print("base_url:", BASE_URL or "(default OpenAI)")
print("ndjson:", ndjson_path)
print("cache:", cache_dir)


client_id: 1696
model: gpt-5.2
base_url: https://aibe.mygreatlearning.com/openai/v1
ndjson: /Users/nicholasp/Personal Coding/JHU/personal finance/artifacts/transactions_enriched.json
cache: /Users/nicholasp/Personal Coding/JHU/personal finance/artifacts/llm_cache


## Load a small set of unique merchants for user 1696

In [4]:
if not ndjson_path.exists():
    raise FileNotFoundError("Missing artifacts/transactions_enriched.json. Run data_processing/clean.ipynb first.")

seen_merchants: set[str] = set()
batch: list[dict] = []

for chunk in pd.read_json(ndjson_path, lines=True, dtype=False, chunksize=CHUNKSIZE):
    chunk = chunk.loc[chunk.get("client_id").astype(str) == str(CLIENT_ID)].copy()
    if chunk.empty:
        continue

    for tx in chunk.to_dict(orient="records"):
        mk = merchant_fingerprint(tx)
        if mk in seen_merchants:
            continue
        seen_merchants.add(mk)
        batch.append(tx)
        if len(batch) >= MAX_UNIQUE_MERCHANTS:
            break
    if len(batch) >= MAX_UNIQUE_MERCHANTS:
        break

if not batch:
    raise RuntimeError(f"No transactions found for client_id={CLIENT_ID}")

print("unique merchants loaded:", len(batch))
pd.DataFrame(
    [
        {
            "id": t.get("id"),
            "amount_usd": t.get("amount_usd"),
            "mcc_code": t.get("mcc_code"),
            "mcc_description": t.get("mcc_description"),
            "merchant_city": t.get("merchant_city"),
            "merchant_state": t.get("merchant_state"),
        }
        for t in batch
    ]
).head(10)


unique merchants loaded: 20


,id,amount_usd,mcc_code,mcc_description,merchant_city,merchant_state
0,7475539,4.02,5812,Eating Places and Restaurants,Merritt Island,FL
1,7475586,9.68,4784,Tolls and Bridge Fees,ONLINE,NaN
2,7475755,3.41,5411,"Grocery Stores, Supermarkets",Merritt Island,FL
3,7477220,9.94,4784,Tolls and Bridge Fees,ONLINE,NaN
4,7477493,-89.00,5541,Service Stations,Merritt Island,FL
5,7481206,91.75,5499,Miscellaneous Food Stores,Merritt Island,FL
6,7483367,2.88,5411,"Grocery Stores, Supermarkets",Merritt Island,FL
7,7494502,40.55,5655,"Sports Apparel, Riding Apparel Stores",Merritt Island,FL
8,7501157,56.00,5541,Service Stations,Merritt Island,FL
9,7501273,55.95,5912,Drug Stores and Pharmacies,Merritt Island,FL


## Make exactly one LLM call (batched) and show results

In [5]:
cache_dir.mkdir(parents=True, exist_ok=True)
cache = Cache(str(cache_dir))

llm_call = make_openai_llm_call(model=LLM_MODEL)

results = categorize_with_cache_and_fallback(
    batch,
    llm_call=llm_call,
    cache=cache,
    confidence_threshold=CONF_THRESHOLD,
    taxonomy=DEFAULT_TAXONOMY,
    cache_key_fn=merchant_fingerprint,
)
cache.close()

print("results:", len(results))
pd.DataFrame(
    [
        {
            "id": r.transaction_id,
            "category_final": r.category_final,
            "source": r.source,
            "confidence": r.confidence,
            "category_llm": r.category_llm,
        }
        for r in results
    ]
).head(20)


results: 20


,id,category_final,source,confidence,category_llm
0,7475539,Dining,llm,0.90,Dining
1,7475586,Transportation,llm,0.85,Transportation
2,7475755,Groceries,llm,0.90,Groceries
3,7477220,Transportation,llm,0.85,Transportation
4,7477493,Transportation,llm,0.70,Transportation
5,7481206,Groceries,llm,0.75,Groceries
6,7483367,Groceries,llm,0.90,Groceries
7,7494502,Shopping,llm,0.85,Shopping
8,7501157,Transportation,llm,0.70,Transportation
9,7501273,Healthcare,llm,0.75,Healthcare
